You’ll be able to:

Evaluate this model alone (DAPT only)

Merge it with your QLoRA SFT adapter (DAPT + SFT)

Use it in RAG setups

🧠 Purpose:
This script performs unsupervised domain-adaptive pretraining (DAPT) using a LoRA adapter on the LLaMA 2 7B model. It trains the model to adapt to the language and structure of the university domain using only raw web content from full_university_data.txt.

Step 2: Imports & Setup
Sets the model ID and file paths.

Prepares essential libraries for dataset loading, tokenization, LoRA configuration, and training.

In [2]:
import os
import torch
import gc
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)
from peft import get_peft_model, LoraConfig, TaskType

model_id = "meta-llama/Llama-2-7b-hf"
data_path = "../data/full_university_data.txt"
output_dir = "../checkpoints/llama2_dapt_lora/"


Step 3: Load and Prepare Dataset
Reads and cleans full_university_data.txt.

Filters out short lines and wraps them into a Hugging Face Dataset.

In [3]:
with open(data_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

lines = [l.strip() for l in raw_text.split("\n") if len(l.strip()) > 30]
dataset = Dataset.from_dict({"text": lines})
print(f"✅ Loaded {len(lines)} samples")


✅ Loaded 6050 samples


Step 4: Tokenizer & 4-bit Base Model
Loads the LLaMA 2 tokenizer.

Clears CUDA memory.

Loads the base LLaMA 2 model in 4-bit quantization using BitsAndBytesConfig to fit it on your 8GB GPU.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=True)
tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    use_auth_token=True
)


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Step 5: Apply LoRA Config for DAPT
Sets LoRA parameters (rank, alpha, dropout).

Targets q_proj and v_proj layers (standard for LLaMA).

Wraps the base model with the LoRA config to create a trainable adapter.

In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)


Step 6: Tokenize the Dataset
Truncates input text to 128 tokens (GPU-friendly).

Uses DataCollatorForLanguageModeling to mask and batch samples (no MLM).

In [6]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map:   0%|          | 0/6050 [00:00<?, ? examples/s]

Step 7: Training Arguments
Defines key training parameters:

Low batch size

FP16

Gradient accumulation

1 epoch (quick and light)

In [7]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none"
)


Step 8: Train & Save Adapter
Runs training using Hugging Face's Trainer.

Saves the LoRA adapter only to ../checkpoints/llama2_dapt_lora/.

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

# Save LoRA adapter
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ LoRA-DAPT complete. Saved to:", output_dir)


C:\Users\berfi\AppData\Local\Temp\ipykernel_47536\1033579740.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  0%|          | 0/756 [00:00<?, ?it/s]

c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\llama\modeling_llama.py:602: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'loss': 2.8111, 'grad_norm': 1.0103728771209717, 'learning_rate': 0.00019735449735449736, 'epoch': 0.01}
{'loss': 2.5714, 'grad_norm': 0.8241586685180664, 'learning_rate': 0.0001947089947089947, 'epoch': 0.03}
{'loss': 2.3978, 'grad_norm': 2.172400951385498, 'learning_rate': 0.00019206349206349208, 'epoch': 0.04}
{'loss': 2.4147, 'grad_norm': 1.3722363710403442, 'learning_rate': 0.00018941798941798943, 'epoch': 0.05}
{'loss': 2.3427, 'grad_norm': 4.232949256896973, 'learning_rate': 0.00018677248677248677, 'epoch': 0.07}
{'loss': 2.2768, 'grad_norm': 2.1028199195861816, 'learning_rate': 0.00018465608465608466, 'epoch': 0.08}
{'loss': 2.1897, 'grad_norm': 1.3773220777511597, 'learning_rate': 0.000182010582010582, 'epoch': 0.09}
{'loss': 2.2137, 'grad_norm': 0.9734731316566467, 'learning_rate': 0.00017936507936507938, 'epoch': 0.11}
{'loss': 2.3384, 'grad_norm': 1.6711632013320923, 'learning_rate': 0.00017671957671957673, 'epoch': 0.12}
{'loss': 2.2371, 'grad_norm': 1.3089354038238525, '

🔄 Where This Code Was Used:
Used in Pipeline P2 (DAPT Only), and also merged with QLoRA in Pipeline P3 (DAPT + SFT) and P4 (Full + RAG).

✅ Suggestions for Improvement

Element	Suggestion
File name	Rename to dapt_lora_finetune.ipynb
Save path	Match structure: checkpoints/dapt_lora/
Epochs	Optionally increase to 2–3 if training longer
Training logs	Add save_logs=True or CSV/JSON export for tracking